# Semana 9: RQ4, exportação das imagens Sentinel-2 do Rio Grande do Sul

Seção 1 de `prereg/plano_rq4.md` (commit `245f811`). **Este notebook não roda nenhum modelo e não lê nenhuma referência de enchente.** Ele só pede ao Google Earth Engine as imagens Sentinel-2 L1C de **2024-05-06 (UTC)** dos tiles **22JDM** (Porto Alegre) e **22JCN** (Encantado) e as guarda no Drive, com o registro do que foi baixado.

Para cada tile são exportados três arquivos na grade de 10 m do próprio granulo:

| Arquivo | Coleção | Conteúdo |
|---|---|---|
| `..._s2.tif` | `COPERNICUS/S2_HARMONIZED` | B2, B3, B4, B8A, B11, B12, uint16. A coleção harmonizada tira o deslocamento de 1.000 que as imagens L1C passam a ter a partir da linha de base 04.00, então os valores ficam na escala dos dados de treino. Pixel sem dado = 0. B8A, B11 e B12 são de 20 m e entram por vizinho mais próximo (padrão do Earth Engine). |
| `..._nuvem.tif` | `COPERNICUS/S2_CLOUD_PROBABILITY` | `probability` do s2cloudless, uint8. Sem dado = 255 (nunca lido como céu limpo). |
| `..._scl.tif` | `COPERNICUS/S2_SR_HARMONIZED` | `SCL` do L2A do mesmo granulo, uint8 (0 = sem dado). |

Se houver mais de um granulo do tile no dia, vale o de menor `CLOUDY_PIXEL_PERCENTAGE`; se nenhum existir, o notebook para (o plano, seção 5, diz o que fazer, e eu decido com você).

| Célula | O que faz |
|---|---|
| 1 | Monta o Drive, clona o repositório, entra no Earth Engine |
| 2 | Lista os granulos de 2024-05-06 de cada tile e escolhe um |
| 3 | Confere que a nuvem e a SCL existem para o mesmo granulo; confere a linha de base e o deslocamento; lê a grade do B2 |
| 4 | Pede as 6 exportações ao Earth Engine e espera terminar (de 20 a 60 minutos) |
| 5 | Move os arquivos para `Mestrado/dados/rs2024/s2`, confere a grade, calcula os SHA-256 e grava `rq4_entradas.json` |

**Antes de rodar:** abra o Colab logada na conta do mestrado (não precisa de GPU) e preencha `PROJETO_GEE` na célula 1 com o ID do seu projeto do Earth Engine. A célula 1 abre a janela de login do Earth Engine na primeira vez. Se a sessão cair durante a célula 4, rode a célula 1 e a 4 de novo: as exportações já pedidas não são repetidas. Se alguma célula der erro, pare e me mande a saída inteira.

Depois da célula 5, me mande `rq4_entradas.json` e `rq4_entradas_sha256.csv` de `resultados/semana9/rq4_entradas`.

In [ ]:
import os, re, json, time, hashlib, glob, shutil, subprocess, sys, csv
import numpy as np

from google.colab import drive
drive.mount('/content/drive')
MEUDRIVE = '/content/drive/MyDrive'
RAIZ = f'{MEUDRIVE}/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

DESTINO = f'{RAIZ}/dados/rs2024/s2'
SAIDA_DRIVE = f'{RAIZ}/resultados/semana9/rq4_entradas'
PASTA_EXPORT = 'rq4_s2'                      # pasta criada pelo Earth Engine na raiz do seu Drive
assert not (os.path.isdir(SAIDA_DRIVE) and os.listdir(SAIDA_DRIVE)), (
    'resultados/semana9/rq4_entradas já tem arquivos. Não sobrescrevo: me avise antes de rodar de novo.')
os.makedirs(DESTINO, exist_ok=True)

REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO_GIT = '/content/repo'
if not os.path.isdir(REPO_GIT):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_GIT], check=True)
else:
    subprocess.run(['git', '-C', REPO_GIT, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO_GIT, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, f'{REPO_GIT}/analysis/rq4')
import s2_exportacao as S
SHA_MODULO = hashlib.sha256(open(f'{REPO_GIT}/analysis/rq4/s2_exportacao.py', 'rb').read()).hexdigest()

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

import ee
PROJETO_GEE = ''                              # PREENCHA: o ID do seu projeto do Earth Engine
assert PROJETO_GEE, 'Preencha PROJETO_GEE com o ID do projeto do Earth Engine.'
try:
    ee.Initialize(project=PROJETO_GEE)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJETO_GEE)

L1C, NUVEM, L2A = 'COPERNICUS/S2_HARMONIZED', 'COPERNICUS/S2_CLOUD_PROBABILITY', 'COPERNICUS/S2_SR_HARMONIZED'
print(f'Repositório no commit {commit[:7]}. Earth Engine pronto (projeto {PROJETO_GEE}).')

In [ ]:
PROPS = ['system:index', 'system:time_start', 'MGRS_TILE', 'PRODUCT_ID', 'GRANULE_ID', 'CLOUDY_PIXEL_PERCENTAGE',
         'PROCESSING_BASELINE', 'SPACECRAFT_NAME', 'SENSING_ORBIT_NUMBER']
ESCOLHIDOS, TODOS = {}, {}
for tile in S.TILES:
    col = (ee.ImageCollection(L1C).filter(ee.Filter.eq('MGRS_TILE', tile))
           .filterDate(S.DATA, '2024-05-07'))                 # dia 2024-05-06 em UTC
    feats = col.getInfo()['features']
    cand = [{'id': f['properties']['system:index'], 'tile': f['properties']['MGRS_TILE'],
             'time_start_ms': f['properties']['system:time_start'],
             'nuvem': f['properties']['CLOUDY_PIXEL_PERCENTAGE'], 'n_candidatos': len(feats),
             'propriedades': {p: f['properties'].get(p) for p in PROPS}} for f in feats]
    TODOS[tile] = [{k: v for k, v in c.items() if k != 'propriedades'} for c in cand]
    print(f'{tile}: {len(cand)} granulo(s) em {S.DATA}')
    for c in cand:
        print('   ', c['id'], f"nuvem {c['nuvem']:.1f}%", c['propriedades']['PROCESSING_BASELINE'], c['propriedades']['SPACECRAFT_NAME'])
    escolhido = S.escolher_granulo(cand, tile)
    escolhido['n_candidatos'] = len(feats)
    S.conferir_linha_base(escolhido['propriedades']['PROCESSING_BASELINE'])
    ESCOLHIDOS[tile] = escolhido
    print(f'   escolhido: {escolhido["id"]}')

In [ ]:
COLECOES = {'l1c': L1C, 'nuvem': NUVEM, 'l2a': L2A}
GRADES, DESLOC = {}, {}
for tile, c in ESCOLHIDOS.items():
    idx = c['id']
    for nome, colecao in (('nuvem', NUVEM), ('l2a', L2A)):
        try:
            ee.Image(f'{colecao}/{idx}').getInfo()
        except ee.EEException as erro:
            raise RuntimeError(f'{tile}: não há {nome} ({colecao}) para o granulo {idx}. Pare e me mande esta saída.') from erro
    proj = ee.Image(f'{L1C}/{idx}').select('B2').projection().getInfo()
    a, b, x0, d, e, y0 = proj['transform']
    assert (a, b, d, e) == (10, 0, 0, -10) and proj['crs'] == S.CRS_ESPERADO[tile], (tile, proj)
    GRADES[tile] = {'crs': proj['crs'], 'transform': proj['transform'], 'origem': (float(x0), float(y0))}

    # Conferência do deslocamento: no mesmo local, o valor bruto menos o harmonizado deve ser 1000 (menos onde o
    # harmonizado foi cortado em 0). Se a coleção bruta não estiver mais disponível, isso fica registrado.
    centro = ee.Geometry.Rectangle([x0 + 5000, y0 - 5500, x0 + 6000, y0 - 4500], proj=proj['crs'], geodesic=False)
    harm = ee.Image(f'{L1C}/{idx}').select('B4').reduceRegion(ee.Reducer.mean(), centro, 10).get('B4').getInfo()
    try:
        bruto = ee.Image(f'COPERNICUS/S2/{idx}').select('B4').reduceRegion(ee.Reducer.mean(), centro, 10).get('B4').getInfo()
        dif = float(bruto) - float(harm)
        DESLOC[tile] = {'situacao': 'conferido' if 900 <= dif <= 1000.5 else 'FORA DO ESPERADO',
                        'b4_bruto_menos_harmonizado': dif, 'b4_harmonizado': float(harm)}
    except Exception as erro:
        DESLOC[tile] = {'situacao': 'colecao bruta indisponivel', 'b4_harmonizado': float(harm), 'erro': type(erro).__name__}
    print(tile, GRADES[tile]['crs'], 'origem', GRADES[tile]['origem'], DESLOC[tile])
    assert DESLOC[tile]['situacao'] != 'FORA DO ESPERADO', f'{tile}: o deslocamento não é o esperado. Pare e me mande esta saída.'

In [ ]:
def imagens(tile):
    idx = ESCOLHIDOS[tile]['id']
    s2 = (ee.Image(f'{L1C}/{idx}').select(list(S.BANDAS)).toUint16().unmask(0))
    nuvem = ee.Image(f'{NUVEM}/{idx}').select('probability').toUint8().unmask(255)
    scl = ee.Image(f'{L2A}/{idx}').select('SCL').toUint8().unmask(0)
    return {'s2': s2, 'nuvem': nuvem, 'scl': scl}

def regiao(tile):
    x0, y0 = GRADES[tile]['origem']
    n = S.LADO * S.PIXEL
    return ee.Geometry.Rectangle([x0, y0 - n, x0 + n, y0], proj=GRADES[tile]['crs'], geodesic=False)

ATIVAS = {t.config.get('description'): t for t in ee.batch.Task.list() if t.state in ('READY', 'RUNNING')}
TAREFAS = {}
for tile in S.TILES:
    for nome, img in imagens(tile).items():
        prefixo = f'{tile}_{ESCOLHIDOS[tile]["id"]}_{nome}'
        ja = glob.glob(f'{MEUDRIVE}/{PASTA_EXPORT}/{prefixo}*.tif') + glob.glob(f'{DESTINO}/{prefixo}*.tif')
        if ja:
            print(f'{prefixo}: já exportado ({len(ja)} arquivo).')
            continue
        descricao = prefixo[:100]
        if descricao in ATIVAS:
            TAREFAS[prefixo] = ATIVAS[descricao]
            print(f'{prefixo}: tarefa já em andamento, não repito.')
            continue
        tarefa = ee.batch.Export.image.toDrive(
            image=img, description=descricao, folder=PASTA_EXPORT, fileNamePrefix=prefixo,
            region=regiao(tile), crs=GRADES[tile]['crs'], crsTransform=GRADES[tile]['transform'],
            maxPixels=int(1e10), fileFormat='GeoTIFF')
        tarefa.start()
        TAREFAS[prefixo] = tarefa
        print(f'{prefixo}: tarefa pedida.')

limite = time.time() + 3 * 3600
ultimo = None
while TAREFAS and time.time() < limite:
    estados = {p: t.status()['state'] for p, t in TAREFAS.items()}
    if estados != ultimo:
        print(time.strftime('%H:%M:%S'), {p.split('_')[-1] + '_' + p[:5]: s for p, s in estados.items()})
        ultimo = estados
    ruins = {p: s for p, s in estados.items() if s in ('FAILED', 'CANCELLED', 'CANCEL_REQUESTED')}
    assert not ruins, f'Exportação falhou: {ruins}. Pare e me mande esta saída.'
    if all(s == 'COMPLETED' for s in estados.values()):
        break
    time.sleep(60)
else:
    if TAREFAS:
        raise RuntimeError('As exportações não terminaram em 3 horas. Rode esta célula de novo mais tarde.')
print('Exportações concluídas.')

In [ ]:
import rasterio

ENTRADAS, LINHAS = [], []
for tile, c in ESCOLHIDOS.items():
    arquivos, grade, zeros = {}, None, None
    for nome in ('s2', 'nuvem', 'scl'):
        prefixo = f'{tile}_{c["id"]}_{nome}'
        origem = glob.glob(f'{MEUDRIVE}/{PASTA_EXPORT}/{prefixo}*.tif')
        destino = glob.glob(f'{DESTINO}/{prefixo}*.tif')
        achados = destino or origem
        assert len(achados) == 1, f'{prefixo}: {len(achados)} arquivos (o esperado é 1; mais de um quer dizer que o Earth Engine dividiu a exportação). Pare e me mande esta saída.'
        caminho = achados[0]
        if not destino:
            novo = f'{DESTINO}/{os.path.basename(caminho)}'
            shutil.move(caminho, novo)
            caminho = novo
        with rasterio.open(caminho) as s:
            perfil = {'height': s.height, 'width': s.width, 'count': s.count, 'dtype': s.dtypes[0],
                      'crs': s.crs.to_string(), 'transform': tuple(s.transform)[:6]}
            if nome == 's2':
                S.conferir_grade(tile, perfil, GRADES[tile]['origem'])
                grade = perfil
                nulos = total = 0
                medias = np.zeros(len(S.BANDAS)); validos = 0
                for r0 in range(0, s.height, 1024):
                    bloco = s.read(window=rasterio.windows.Window(0, r0, s.width, min(1024, s.height - r0))).astype(np.float64)
                    tudo_zero = (bloco == 0).all(axis=0)
                    nulos += int(tudo_zero.sum()); total += tudo_zero.size
                    medias += bloco[:, ~tudo_zero].sum(axis=1); validos += int((~tudo_zero).sum())
                zeros = nulos / total
                medias = medias / max(validos, 1)
                print(f'{tile}: {zeros:.4f} dos pixels com as 6 bandas em 0; médias dos válidos', dict(zip(S.BANDAS, np.round(medias, 1))))
            else:
                assert (s.height, s.width) == (S.LADO, S.LADO) and s.count == 1 and s.dtypes[0] == 'uint8', (nome, perfil)
                assert tuple(s.transform)[:6] == tuple(grade['transform']) if grade else True
        arquivos[nome] = {'arquivo': os.path.basename(caminho), 'bytes': os.path.getsize(caminho), 'sha256': sha256_arquivo(caminho)}
        LINHAS.append([os.path.basename(caminho), arquivos[nome]['bytes'], arquivos[nome]['sha256']])
    ENTRADAS.append(S.registro_entrada(
        tile, c, c['propriedades'], COLECOES, arquivos,
        {'crs': grade['crs'], 'transform': list(grade['transform']), 'altura': grade['height'], 'largura': grade['width']},
        zeros, DESLOC[tile]))

os.makedirs(SAIDA_DRIVE, exist_ok=True)
INFO = {'plano': 'prereg/plano_rq4.md, secao 1', 'commit_repositorio': commit, 's2_exportacao_sha256': SHA_MODULO,
        'data_sensing_utc': S.DATA, 'candidatos_no_dia': TODOS, 'entradas': ENTRADAS,
        'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA_DRIVE}/rq4_entradas.json', 'w'), indent=2, ensure_ascii=False)
with open(f'{SAIDA_DRIVE}/rq4_entradas_sha256.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['arquivo', 'bytes', 'sha256']); w.writerows(LINHAS)
print(open(f'{SAIDA_DRIVE}/rq4_entradas_sha256.csv').read())
print(f'Registro em {SAIDA_DRIVE}. Nenhum modelo foi rodado.')